In [32]:
from dotenv import load_dotenv
import os
import pandas as pd
import mysql.connector
import warnings
warnings.filterwarnings('ignore')


In [3]:
load_dotenv()

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASS"),
    database=os.getenv("DB_DB")
)

query = """
select *
from orders o
join deliveries d on o.order_id = d.order_id
join riders r on o.rider_id = r.rider_id
"""

df = pd.read_sql(query, conn)

C:\Users\Ashok\AppData\Local\Temp\ipykernel_12552\3274090984.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [27]:
df.head()

,order_id,order_date,time_ordered,store_id,rider_id,product_id,num_items,drop_lat,drop_lon,store_lat,...,rating,category,weight_class,time_order_picked,distance_km,traffic,weather,store_load,rider_batch_count,delivery_time_min
0,ORD_00000,2026-01-29,0 days 08:53:00,STR_00066,RDR_00978,PRD_00021,3,12.8649,80.1900,12.8993,...,4.54,grocery,light\r,0 days 08:59:00,7.64,high,clear,39,1,20.49
1,ORD_00001,2026-01-09,0 days 13:19:00,STR_00026,RDR_00738,PRD_00010,4,12.8650,80.0658,12.9199,...,4.25,electronics,medium\r,0 days 13:28:00,7.42,medium,clear,48,1,16.76
2,ORD_00002,2026-01-29,0 days 19:33:00,STR_00060,RDR_00090,PRD_00007,4,13.1419,80.2216,12.9860,...,4.10,electronics,medium\r,0 days 19:38:00,2.79,high,clear,22,1,11.30
3,ORD_00003,2026-01-09,0 days 18:52:00,STR_00047,RDR_00404,PRD_00000,2,13.0420,80.1817,13.0320,...,3.60,bulky,heavy\r,0 days 18:55:00,3.40,medium,clear,34,1,14.28
4,ORD_00004,2026-01-04,0 days 14:27:00,STR_00022,RDR_00310,PRD_00000,1,12.9582,80.0535,12.9523,...,4.72,bulky,heavy\r,0 days 14:32:00,4.58,low,clear,35,1,16.27


In [51]:
df = df.loc[:, ~df.columns.duplicated()]

In [5]:
df.isnull().sum()

order_id             0
order_date           0
time_ordered         0
store_id             0
rider_id             0
product_id           0
num_items            0
drop_lat             0
drop_lon             0
store_lat            0
store_lon            0
vehicle_type         0
rating               0
category             0
weight_class         0
order_id             0
time_order_picked    0
distance_km          0
traffic              0
weather              0
store_load           0
rider_batch_count    0
delivery_time_min    0
rider_id             0
vehicle_type         0
rating               0
dtype: int64

In [28]:
df.duplicated().sum()

np.int64(0)

In [29]:
df.describe()

,time_ordered,num_items,drop_lat,drop_lon,store_lat,store_lon,rating,time_order_picked,distance_km,store_load,rider_batch_count,delivery_time_min
count,100000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000,100000.000000,100000.000000,100000.000000,100000.000000
mean,0 days 13:30:54.823200,3.007090,13.025248,80.199922,13.013814,80.199025,4.253215,0 days 13:35:43.637400,3.629235,27.023570,1.403360,13.352671
std,0 days 05:23:38.652424340,1.414935,0.100897,0.086648,0.103402,0.087547,0.432182,0 days 05:23:59.904332072,1.943677,12.961646,0.666097,5.016015
min,0 days 00:00:00,1.000000,12.850000,80.050000,12.851900,80.052100,3.500000,0 days 00:00:00,1.000000,5.000000,1.000000,8.000000
25%,0 days 09:07:00,2.000000,12.938475,80.124800,12.914700,80.122600,3.870000,0 days 09:12:00,2.000000,16.000000,1.000000,8.360000
50%,0 days 13:40:00,3.000000,13.025000,80.200100,13.009600,80.200800,4.270000,0 days 13:45:00,3.020000,27.000000,1.000000,12.460000
75%,0 days 18:14:00,4.000000,13.112400,80.274700,13.105400,80.278500,4.620000,0 days 18:19:00,5.150000,38.000000,2.000000,16.770000
max,0 days 23:59:00,5.000000,13.200000,80.350000,13.195400,80.345700,5.000000,0 days 23:59:00,8.000000,49.000000,3.000000,35.000000


## feature selection

In [8]:
df[['distance_km', 'delivery_time_min']].corr()

,distance_km,delivery_time_min
distance_km,1.000000,0.785055
delivery_time_min,0.785055,1.000000


In [9]:
df[['num_items', 'delivery_time_min']].corr()

,num_items,delivery_time_min
num_items,1.000000,-0.000934
delivery_time_min,-0.000934,1.000000


In [30]:
corr = df.corr(numeric_only=True)['delivery_time_min'].sort_values(ascending=False)
print(corr)

delivery_time_min    1.000000
distance_km          0.785055
rider_batch_count    0.335255
drop_lat             0.001226
store_load           0.000812
store_lat            0.000495
num_items           -0.000934
store_lon           -0.002009
drop_lon            -0.002959
rating              -0.095536
Name: delivery_time_min, dtype: float64


In [13]:
df.nunique().sort_values()

category                  3
weight_class              3
vehicle_type              3
weather                   3
vehicle_type              3
rider_batch_count         3
traffic                   3
num_items                 5
order_date               30
store_load               45
product_id               50
store_lon                98
store_lat                99
store_id                100
rating                  151
rating                  151
distance_km             701
rider_id               1000
rider_id               1000
time_order_picked      1440
time_ordered           1440
delivery_time_min      2311
drop_lon               3001
drop_lat               3501
order_id             100000
order_id             100000
dtype: int64

In [23]:
print(df.columns)

Index(['order_id', 'order_date', 'time_ordered', 'store_id', 'rider_id',
       'product_id', 'num_items', 'drop_lat', 'drop_lon', 'store_lat',
       'store_lon', 'vehicle_type', 'rating', 'category', 'weight_class',
       'order_id', 'time_order_picked', 'distance_km', 'traffic', 'weather',
       'store_load', 'rider_batch_count', 'delivery_time_min', 'rider_id',
       'vehicle_type', 'rating'],
      dtype='object')


In [14]:
df.groupby('traffic')['delivery_time_min'].mean()

traffic
high      15.083603
low       12.612085
medium    13.359562
Name: delivery_time_min, dtype: float64

In [24]:
df = df.loc[:, ~df.columns.duplicated()]

In [ ]:
df.groupby('category')['delivery_time_min'].mean()

category
bulky          17.199503
electronics    12.847462
grocery        12.825776
Name: delivery_time_min, dtype: float64

In [20]:
df.groupby('category')['delivery_time_min'].mean()

category
bulky          17.199503
electronics    12.847462
grocery        12.825776
Name: delivery_time_min, dtype: float64

In [25]:
df.groupby('vehicle_type')['delivery_time_min'].mean()

vehicle_type
electric      13.757212
motorcycle    13.144242
scooter       13.402867
Name: delivery_time_min, dtype: float64

In [26]:
df.groupby('weight_class')['delivery_time_min'].mean()

weight_class
heavy\r     17.199503
light\r     12.825776
medium\r    12.847462
Name: delivery_time_min, dtype: float64

## Feature Extraction

In [33]:
traffic_map = {'low':1, 'medium':2, 'high':3}
df['traffic_level'] = df['traffic'].map(traffic_map)

df['traffic_batch'] = df['traffic_level'] * df['rider_batch_count']

In [34]:
df[['traffic_batch', 'delivery_time_min']].corr()

,traffic_batch,delivery_time_min
traffic_batch,1.000000,0.360969
delivery_time_min,0.360969,1.000000


In [39]:
df['order_hour'] =df['order_hour'] = df['time_ordered'].dt.components.hours

df['is_peak'] = df['order_hour'].isin([12,13,18,19]).astype(int)

In [41]:
df[['is_peak','delivery_time_min']].corr()

,is_peak,delivery_time_min
is_peak,1.000000,0.059027
delivery_time_min,0.059027,1.000000


In [42]:
df['distance_batch'] = df['distance_km'] * df['rider_batch_count']

In [43]:
df[['distance_batch','delivery_time_min']].corr()

,distance_batch,delivery_time_min
distance_batch,1.000000,0.785089
delivery_time_min,0.785089,1.000000


In [44]:
df['hourly_demand'] = df.groupby('order_hour')['order_id'].transform('count')

In [45]:
df[['hourly_demand','delivery_time_min']].corr()

,hourly_demand,delivery_time_min
hourly_demand,1.000000,0.083119
delivery_time_min,0.083119,1.000000


In [46]:
df['distance_bucket'] = pd.cut(
    df['distance_km'],
    bins=[0,2,5,10],
    labels=['short','medium','long']
)

In [50]:
df.groupby('distance_bucket')['delivery_time_min'].mean()

distance_bucket
short      8.978020
medium    12.566018
long      18.889062
Name: delivery_time_min, dtype: float64

In [60]:
features = [
    'distance_km',
    'rider_batch_count',
    'traffic',
    'weather',
    'traffic_batch',
    'is_peak',
    'order_hour',
    'distance_bucket',
    'weight_class',
    'delivery_time_min'
]

In [61]:
final_features = df[features]

In [66]:
final_features.head()

,distance_km,rider_batch_count,traffic,weather,traffic_batch,is_peak,order_hour,distance_bucket,weight_class,delivery_time_min
0,7.64,1,high,clear,3,0,8,long,light\r,20.49
1,7.42,1,medium,clear,2,1,13,long,medium\r,16.76
2,2.79,1,high,clear,3,1,19,medium,medium\r,11.30
3,3.40,1,medium,clear,2,1,18,medium,heavy\r,14.28
4,4.58,1,low,clear,1,0,14,medium,heavy\r,16.27


In [68]:
final_features.to_parquet(r'D:\zepto_ETA_prediction\data\features\final_features.parquet')